# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fin1806/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Build the feature vector

One training row represents one content item for one client at the cutoff of 31 May 2026. The features use 2 to 31 May. The label uses 1 to 30 June. Both windows need 30 observed daily rows. Pages with fewer than 100 impressions in the first window are left out because a small change on a very low count is hard to interpret.

The code reads only the May and June warehouse partitions. It removes repeated rows with the same GSC values and stops if a date, client, and content key still has conflicting values. It prints counts only, not client or page IDs.

In [1]:
import os
import getpass
from datetime import date, timedelta
import duckdb
import numpy as np

hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
            + hf_token.replace("'", "''") + "')")
del hf_token

label_end = date(2026, 6, 30)
cutoff = label_end - timedelta(days=30)
feature_start = cutoff - timedelta(days=29)
assert (label_end - feature_start).days + 1 == 60

warehouse = 'hf://datasets/FlyRank/internship-warehouse'
months = sorted({feature_start.strftime('%Y-%m'), label_end.strftime('%Y-%m')})
daily_parts = [
    f'{warehouse}/fact_content_daily_performance/month={month}/*.parquet'
    for month in months
]
daily_source = f'read_parquet({daily_parts!r})'

# Read the two monthly partitions once, then do the remaining checks locally.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE window_raw AS
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM {daily_source}
    WHERE report_date BETWEEN DATE '{feature_start}' AND DATE '{label_end}'
""")
raw_rows = con.sql('SELECT COUNT(*) FROM window_raw').fetchone()[0]
con.execute('CREATE OR REPLACE TEMP TABLE window_daily AS SELECT DISTINCT * FROM window_raw')
daily_rows = con.sql('SELECT COUNT(*) FROM window_daily').fetchone()[0]
missing_keys = con.sql("""
    SELECT COUNT(*) FROM window_daily
    WHERE report_date IS NULL OR client_hash_id IS NULL OR content_hash_id IS NULL
""").fetchone()[0]
conflicting_keys = con.sql("""
    SELECT COUNT(*) FROM (
        SELECT 1 FROM window_daily
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f'Window: {feature_start} to {cutoff} for features; '
      f'{cutoff + timedelta(days=1)} to {label_end} for the label')
print(f'Window rows: {raw_rows:,}; repeated GSC rows removed: {raw_rows - daily_rows:,}')
assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
assert conflicting_keys == 0, f'{conflicting_keys:,} daily keys have different GSC values'
con.execute('DROP TABLE window_raw')  # release the larger local staging table

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE page_windows AS
    WITH per_page AS (
        SELECT client_hash_id, content_hash_id,
               COUNT(*) FILTER (WHERE report_date <= DATE '{cutoff}') AS prior_days,
               COUNT(*) FILTER (WHERE report_date > DATE '{cutoff}') AS future_days,
               COUNT(*) FILTER (WHERE report_date <= DATE '{cutoff}' AND
                                (gsc_impressions IS NULL OR gsc_clicks IS NULL)) AS missing_prior,
               COUNT(*) FILTER (WHERE report_date > DATE '{cutoff}' AND
                                gsc_impressions IS NULL) AS missing_future,
               SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{cutoff}') AS prior30_impressions,
               SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '{cutoff}') AS prior30_clicks,
               SUM(gsc_impressions) FILTER (WHERE report_date > DATE '{cutoff}') AS future30_impressions,
               SUM(CASE WHEN report_date <= DATE '{cutoff}' AND gsc_avg_position IS NOT NULL
                             AND gsc_impressions > 0
                        THEN gsc_avg_position * gsc_impressions ELSE 0 END)
               / NULLIF(SUM(CASE WHEN report_date <= DATE '{cutoff}'
                                     AND gsc_avg_position IS NOT NULL AND gsc_impressions > 0
                                 THEN gsc_impressions ELSE 0 END), 0) AS prior30_avg_position
        FROM window_daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT * FROM per_page
    WHERE prior_days = 30 AND future_days = 30
      AND missing_prior = 0 AND missing_future = 0
      AND prior30_impressions >= 100
""")

feature_frame = con.sql("""
    SELECT client_hash_id, content_hash_id,
           prior30_impressions, prior30_clicks, prior30_avg_position,
           future30_impressions,
           LN(1 + prior30_impressions) AS log_prior30_impressions,
           LN(1 + prior30_clicks) AS log_prior30_clicks,
           CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
           COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
           CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing,
           CASE WHEN future30_impressions < 0.8 * prior30_impressions
                THEN 1 ELSE 0 END AS future_decline_label
    FROM page_windows
""").df()
feature_columns = [
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing'
]
assert len(feature_frame) > 0, 'No pages met the 60-day coverage and impression rules'
print(f'Eligible client-content rows: {len(feature_frame):,}')
print(f'Observed decline-label rate: {feature_frame.future_decline_label.mean():.1%}')

Window: 2026-05-02 to 2026-05-31 for features; 2026-06-01 to 2026-06-30 for the label
Window rows: 23,017,813; repeated GSC rows removed: 6,390
Eligible client-content rows: 102,532
Observed decline-label rate: 65.8%


## 2. Feature notes

The model inputs are five numeric values. log_prior30_impressions and log_prior30_clicks reduce the effect of very large pages. prior30_ctr is clicks divided by impressions in the same 30-day feature window. prior30_avg_position_filled is an impression-weighted average position; zero is only a placeholder when position is missing, and position_missing marks those rows. Zero is not treated as a real search rank.

All five values are available by the 31 May cutoff. Client and content IDs stay outside the model inputs, so there are no categorical fields to encode. Missing impressions or clicks make a page ineligible instead of being filled with zero. The checks below report missing inputs without displaying any page data.

In [2]:
X = feature_frame[feature_columns].copy()
y = feature_frame['future_decline_label'].astype(int)
missing_features = X.isna().sum()
print('Model feature columns:', ', '.join(feature_columns))
print(f'Rows with missing average position: {int(feature_frame.position_missing.sum()):,}')
print(f'Missing values left in the model inputs: {int(missing_features.sum()):,}')
print('Categorical model inputs: none')
assert missing_features.sum() == 0
assert set(np.unique(y)).issubset({0, 1})

Model feature columns: log_prior30_impressions, log_prior30_clicks, prior30_ctr, prior30_avg_position_filled, position_missing
Rows with missing average position: 0
Missing values left in the model inputs: 0
Categorical model inputs: none


## 3. The leakage hunt

The label depends on June impressions, so June values and anything calculated from them cannot be model inputs. The five feature columns use May only. The query table's fixed 90-day values are also left out because that window can include June. No product score or refresh flag is used.

The check below keeps whole clients on one side of the split. It reports the label base rate and compares a small tree using the allowed features with a deliberately leaky tree that receives the June-to-May impression ratio. The leaky result is only a check that the audit can detect the answer being fed to a model. A single cutoff and client split are not a final time-based validation.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

forbidden = {
    'future30_impressions', 'future_decline_label', 'client_hash_id',
    'content_hash_id', 'report_date', 'ga4_data_available'
}
assert not forbidden.intersection(feature_columns)
assert not any('query' in name or 'url' in name or 'hash' in name
               for name in feature_columns)

groups = feature_frame['client_hash_id']
assert groups.nunique() >= 2, 'Need at least two clients for a grouped split'
train_idx, test_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
).split(X, y, groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

honest_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42)
honest_tree.fit(X.iloc[train_idx], y.iloc[train_idx])
honest_accuracy = accuracy_score(y.iloc[test_idx], honest_tree.predict(X.iloc[test_idx]))
majority_class = int(y.iloc[train_idx].mean() >= 0.5)
base_accuracy = accuracy_score(y.iloc[test_idx],
                               np.full(len(test_idx), majority_class))

# Deliberate leakage test: this ratio directly encodes the future-window label.
X_leaky = X.assign(future_to_prior_ratio=(
    feature_frame['future30_impressions'] / feature_frame['prior30_impressions']
))
leaky_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
leaky_tree.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_accuracy = accuracy_score(y.iloc[test_idx],
                                leaky_tree.predict(X_leaky.iloc[test_idx]))
print(f'Client groups: {groups.iloc[train_idx].nunique()} train, '
      f'{groups.iloc[test_idx].nunique()} test')
print(f'Train label rate: {y.iloc[train_idx].mean():.1%}; '
      f'test label rate: {y.iloc[test_idx].mean():.1%}')
print(f'Majority baseline accuracy: {base_accuracy:.3f}')
print(f'Allowed-feature tree accuracy: {honest_accuracy:.3f}')
print(f'Deliberately leaky tree accuracy: {leaky_accuracy:.3f}')
if leaky_accuracy < 0.99:
    print('Leakage check needs review: the future ratio did not nearly recover the label')

Client groups: 31 train, 11 test
Train label rate: 65.8%; test label rate: 65.5%
Majority baseline accuracy: 0.655
Allowed-feature tree accuracy: 0.655
Deliberately leaky tree accuracy: 1.000


## 4. What is excluded and why

The model does not receive client or content IDs. They identify rows and keep clients grouped during testing, but they are not page-quality signals. June impressions, the decline label, and the June-to-May ratio are excluded because they reveal the outcome. The 90-day query table is excluded because its window may overlap June. GA4 fields are excluded because this version uses GSC-only inputs and some GA4 history is unavailable or zero-filled. Product scores and refresh flags are excluded because they can encode earlier decisions.

The feature list below is the one used by the allowed-feature tree. The leaky ratio exists only inside the separate audit test.

In [4]:
excluded_patterns = ('future', 'label', 'hash', 'url', 'query', 'ga4', 'refresh', 'score')
assert all(not any(pattern in name for pattern in excluded_patterns)
           for name in feature_columns)
assert list(X.columns) == feature_columns
print('Final model inputs:', ', '.join(feature_columns))
print('Excluded fields are absent from the final model inputs')

Final model inputs: log_prior30_impressions, log_prior30_clicks, prior30_ctr, prior30_avg_position_filled, position_missing
Excluded fields are absent from the final model inputs


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.